In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv()
os.environ['TAVILY_API_KEY']= os.getenv("TAVILY_API_KEY")


In [2]:
model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)

In [3]:
from pydantic import BaseModel,Field


In [4]:
class Movie(BaseModel):
    title:str=Field(description="the title of movie")
    year:int=Field(description="the year of movie")
    director:str=Field(description="the director of movie")
    rating:float=Field(description="rate movie out of 10")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000234970B9010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000234970B9A90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the titl

In [6]:
model.invoke("provide the details about movie inception")

AIMessage(content='**Inception** (2010) – a science‑fiction thriller directed by Christopher\u202fNolan\n\n| Item | Details |\n|------|---------|\n| **Release dates** | •\u202fUnited States & Canada – **July\u202f16, 2010** (limited)  <br>•\u202fUnited States & Canada – **July\u202f16, 2010** (wide)  <br>•\u202fWorldwide – **July\u202f16, 2010** |\n| **Runtime** | **148 minutes** |\n| **Genre** | Science‑fiction, Thriller, Action |\n| **Language** | English |\n| **Budget** | **$160\u202fmillion** |\n| **Box office** | **$828\u202fmillion** worldwide (≈\u202f$157\u202fmillion domestic, $671\u202fmillion international) |\n| **Production companies** | Warner Bros. Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Wally\u202fPfister |\n| **Edited by** | Lee\u202fSmith |\n| **Production designer** | Guy\u202fBennett |\n| **Set decorator** | Paul\u202fW.\u202fH.\u202fK.\u202fS.\u202fR.\u202fS.\u202fJ.\u202fS.\u202f

In [7]:
model_with_structure.invoke("provide the details about movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

message output alongside parsed structure

In [9]:
class Movie(BaseModel):
    title:str=Field(description="the title of movie")
    year:int=Field(description="the year of movie")
    director:str=Field(description="the director of movie")
    rating:float=Field(description="rate movie out of 10")
    
    
    
structure_model = model.with_structured_output(Movie,include_raw=True)

response = structure_model.invoke("provide details about movie inception")
response



{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie Inception. The user says "provide details about movie inception". The developer provided a function "Movie" which presumably can be called to get details about a movie. The user didn\'t specify a function call. But the instructions say: "When you want to use a tool, you must do so by calling the function." We should call the function with appropriate arguments. The function signature: Movie({director, rating, title, year}). But the function likely expects the movie details? Actually the function might return details for the movie. But the arguments are the director, rating, title, year. Wait: the function is defined as a function that returns some details about a movie. But the arguments are the director, rating, title, year. That seems odd: the function requires these details to be passed in. Maybe it\'s for retrieving data? But the function signature indicates you must su

nested structure

In [10]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None,description="budget of movie in million usd$")
    
    
    

In [11]:
structure_model = model.with_structured_output(MovieDetails,include_raw=True)

response = structure_model.invoke("provide details about movie inception")
response



{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We need to call the function with the required arguments. The function signature: MovieDetails with budget? number|null, cast array of objects with name and role, genres array of strings, title string, year number. Provide details. We need to supply the data. Provide cast: Leonardo DiCaprio as Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page as Ariadne, Tom Hardy as Eames, Ken Watanabe as Saito, etc. Provide budget: $160 million. Year: 2010. Genres: Action, Sci-Fi, Thriller. Provide details.', 'tool_calls': [{'id': 'fc_d025e1dc-7bc1-460f-8197-c67c07ea554e', 'function': {'arguments': '{"budget":160000000,"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Cillian Murphy","role":"Robert Fischer"},{"na